Librairies

In [ ]:
!pip install beautifulsoup4 requests

In [18]:
from bs4 import BeautifulSoup
import os
import requests
import bs4

req = requests.get("https://www.federalreserve.gov/monetarypolicy/fomccalendars.htm")

In [ ]:
soup = BeautifulSoup(req.content,'html.parser')
with open('data/page.html','w',encoding='utf-8') as pge:
  pge.write(soup.prettify())

In [ ]:
article = soup.find('div',{'id':'article'})

In [7]:
panels = article.find_all('div',{'class':'panel panel-default'})

EXTRACTION DES URLS

In [28]:
minutes = []
statements = []
for panel in panels:
  children = list(panel.children)
  children = [c for c in children if not isinstance(c,bs4.element.NavigableString) ]
  i = 0
  for child in children[1:-1]:
    urls = child.find_all('a')
    urls = [a['href'] for a in urls if a['href'].endswith('.htm') and '/fomcpresconf' not in a['href']]
    minutes += [a for a in urls if '/fomcminutes' in a]
    statements += [a for a in urls if 'a.htm' in a]
print(f"minutes: {len(minutes)}")
print(f"statements: {len(statements)}")

minutes: 47
statements: 48


In [31]:
root_url = 'https://www.federalreserve.gov'

In [38]:
import shutil
shutil.rmtree('minutes')
shutil.rmtree('statements')

SAUVEGARDE DES FICHIERS .HTM ET EXTRACTION DU TEXTE

In [39]:
import os
os.makedirs('data/minutes',exist_ok=True) ## Chemin vers le dossier contenant les fichiers.htm des minutes
os.makedirs('data/statements',exist_ok=True) ## Chemin vers le dossier contenant les fichiers.htm des statements
results = []
for m in minutes :
  url = root_url + m if m.startswith('/') else m
  req = requests.get(url)
  soup = BeautifulSoup(req.content,'html.parser')
  with open(f'data/minutes/{m.split("/")[-1]}','w',encoding='utf-8') as f:
    f.write(soup.prettify())
  article = soup.find('div',{'id':'article'})
  #if name=='minute' and file.endswith('fomcminutes20220615.htm'):
  paragraphs = article.find_all('p')
  file = url.split('/')[-1]
  stop = [i for i,p in enumerate(paragraphs) if "______" in p.get_text().strip()]
  print(stop)
  stop = stop[0]
  txt = ""
  for p in paragraphs[:stop]:
      for a in p.find_all("a"):
          a.decompose()
      txt += p.get_text(strip=True) +'\n'
  year = file.split('/')[-1].split('.')[0][-8:-4]
  month = file.split('/')[-1].split('.')[0][-4:-2]
  source = f'minute {month} {year}'
  results.append({'type':'minute','source':source,'content':txt})

for s in statements :
  url = root_url + s if s.startswith('/') else s
  req = requests.get(url)
  soup = BeautifulSoup(req.content,'html.parser')
  with open(f'data/statements/{s.split("/")[-1]}','w',encoding='utf-8') as f:
    f.write(soup.prettify())
  article = soup.find('div',{'class':'col-xs-12 col-sm-8 col-md-8'})
  paragraphs = article.find_all('p')
  file = url.split('/')[-1]
  print(file)
  txt = ""
  for p in paragraphs[:-2]:
      for a in p.find_all("a"):
          a.decompose()
      txt += p.get_text(strip=True) +'\n'
  year = file.split('/')[-1].split('a.')[0][-8:-4]
  month = file.split('/')[-1].split('a.')[0][-4:-2]
  source = f'statement {month} {year}'
  results.append({'type':'statement','source':source,'content':txt})





[137]
[117]
[136]
[111]
[126]
[118]
[125]
[143]
[137]
[144]
[145]
[122]
[119]
[148]
[132]
[137]
[132]
[124]
[114]
[123]
[112]
[129]
[118]
[139]
[119]
[131]
[120]
[116]
[114]
[122]
[116]
[241]
[105]
[97]
[97]
[99]
[124]
[122]
[127]
[229]
[108]
[102]
[112]
[111]
[99]
[104]
[113]
monetary20260128a.htm
monetary20260318a.htm
monetary20260429a.htm
monetary20260617a.htm
monetary20260729a.htm
monetary20260916a.htm
monetary20261028a.htm
monetary20250129a.htm
monetary20250319a.htm
monetary20250507a.htm
monetary20250618a.htm
monetary20250730a.htm
monetary20250822a.htm
monetary20250917a.htm
monetary20251029a.htm
monetary20251210a.htm
monetary20240131a.htm
monetary20240320a.htm
monetary20240501a.htm
monetary20240612a.htm
monetary20240731a.htm
monetary20240918a.htm
monetary20241107a.htm
monetary20241218a.htm
monetary20230201a.htm
monetary20230322a.htm
monetary20230503a.htm
monetary20230614a.htm
monetary20230726a.htm
monetary20230920a.htm
monetary20231101a.htm
monetary20231213a.htm
monetary20220126a.

In [40]:
source

'statement 12 2021'

SAUVEGARDE texts.json.
Ce fichier sera envoyé à un LLM (CLAUDE) pour générer des Auestions réponses de type SQuAD

In [44]:
import json
os.makedirs('data',exist_ok=True)
with open('data/texts.json','w',encoding='utf-8') as tjx:
    tjx.write(json.dumps(results,indent=4, ensure_ascii=False))